# Notebook 2 — Tipos de Séries Temporais

**Disciplina:** Estatística II — Ciência de Dados
**Fatec — 2026**

---

## Objetivos de aprendizagem

Ao final deste notebook, você será capaz de:

1. Distinguir séries **estacionárias × não estacionárias** e interpretar a **ACF**.
2. Distinguir séries **lineares × não lineares**.
3. Identificar **sazonalidade** em dados reais.
4. Diferenciar séries **univariadas × multivariadas** e investigar correlações.
5. **Diagnosticar** uma série temporal brasileira real.

## Metodologia

Ao longo deste notebook, você vai encontrar:

- 💡 **Perguntas investigativas** após cada visualização — não pule!
- 🐞 **Debug challenges** — código propositalmente quebrado para você consertar.
- 📊 **Dados reais brasileiros** — IPCA, dólar e desemprego direto do BCB.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import linregress
from statsmodels.nonparametric.smoothers_lowess import lowess

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.random.seed(123)
print("Setup OK")

---

## 1. Estacionária × Não estacionária

Uma série é **estacionária** quando suas propriedades estatísticas (média, variância) **não mudam com o tempo**.

| Estacionária | Não estacionária |
|--------------|------------------|
| Média constante | Média varia |
| Variância constante | Variância varia |
| Sem tendência | Com tendência/ciclo |

> **Por que importa?** A maioria dos modelos de previsão (ARMA, ARIMA) **exige estacionariedade**. Séries não estacionárias geralmente precisam ser transformadas antes.

### 1.1 Predição antes de executar

Vamos gerar duas séries:
- `serie_est`: números aleatórios com média 10 e desvio 2 (**estacionária**)
- `serie_nao_est`: passos aleatórios acumulados — uma "caminhada aleatória" (**não estacionária**)

✏️ **Antes de rodar:** qual das duas você acha que vai subir indefinidamente? E qual vai oscilar em torno de um valor constante?

_Sua predição:_ _____________________________________________

In [ ]:
tempo = np.arange(1, 101)

# Série estacionária: média fixa, variância fixa
serie_est = np.random.normal(loc=10, scale=2, size=100)

# Série não estacionária: passos aleatórios (random walk)
passos = np.random.normal(loc=0.5, scale=2, size=100)
serie_nao_est = np.cumsum(passos)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(tempo, serie_est, color='steelblue')
ax1.axhline(np.mean(serie_est), color='darkred', linestyle='--', label=f'Média = {np.mean(serie_est):.1f}')
ax1.set_title('Série Estacionária')
ax1.set_xlabel('Tempo'); ax1.legend()

ax2.plot(tempo, serie_nao_est, color='darkorange')
ax2.axhline(np.mean(serie_nao_est), color='darkred', linestyle='--', label=f'Média = {np.mean(serie_nao_est):.1f}')
ax2.set_title('Série Não Estacionária')
ax2.set_xlabel('Tempo'); ax2.legend()

plt.tight_layout()
plt.show()

### 🧠 Interprete

1. Na série estacionária, a linha da média faz sentido como "valor típico"? E na não estacionária?

   _______________________________________________________________

2. Se você **dividisse** a série não estacionária em dois pedaços (primeiros 50 pontos e últimos 50), a média seria igual nos dois pedaços?

   _______________________________________________________________

3. Verifique rodando o código abaixo:

In [ ]:
print("Série Estacionária:")
print(f"  Média 1ª metade: {serie_est[:50].mean():.2f}   2ª metade: {serie_est[50:].mean():.2f}")

print("\nSérie Não Estacionária:")
print(f"  Média 1ª metade: {serie_nao_est[:50].mean():.2f}   2ª metade: {serie_nao_est[50:].mean():.2f}")

### 1.2 Função de Autocorrelação (ACF)

A **ACF** mede se os valores **passados ajudam a prever** os valores futuros da própria série (defasagens, ou *lags*).

| Padrão na ACF | Significado |
|---|---|
| Queda rápida e oscilação em torno de 0 | **Estacionária** (sem memória de longo prazo) |
| Decaimento lento | **Não estacionária** (tendência persistente) |
| Picos regulares em lags fixos | **Sazonalidade** |

### ✏️ Predição antes de ver a ACF

Desenhe no caderno (ou descreva em palavras) como você acha que vai ser a ACF de cada uma das duas séries acima.

_Série estacionária:_ _______________________________________________

_Série não estacionária:_ _______________________________________________

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(serie_est, ax=ax1, lags=20, title='ACF — Série Estacionária')
plot_acf(serie_nao_est, ax=ax2, lags=20, title='ACF — Série Não Estacionária')

plt.tight_layout()
plt.show()

### 🎯 Confirmando a teoria

Na série estacionária, a ACF **cai rapidamente** para valores próximos de zero — os valores passados **não** ajudam muito a prever o futuro.

Na série não estacionária, a ACF **decai lentamente** — os valores estão **fortemente correlacionados com o passado** (há memória de longo prazo).

### 💡 Caso real: séries do BCB

In [ ]:
# Carrega dólar mensal (série 3698 do BCB: taxa média mensal de câmbio)
# Série 1 é diária; vamos usar a 3698 que já é mensal
url_dolar = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.3698/dados?formato=json&dataInicial=01/01/2010&dataFinal=31/12/2024"

dolar = pd.read_json(url_dolar)
dolar['data'] = pd.to_datetime(dolar['data'], format='%d/%m/%Y')
dolar = dolar.set_index('data')
dolar.columns = ['USD/BRL']

print(f"{len(dolar)} observações")
dolar.tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(dolar.index, dolar['USD/BRL'], color='darkgreen')
ax1.set_title('Cotação Dólar/Real (mensal)')
ax1.set_xlabel('Ano'); ax1.set_ylabel('R$')

plot_acf(dolar['USD/BRL'], ax=ax2, lags=24, title='ACF — Dólar/Real')

plt.tight_layout()
plt.show()

### ✏️ Classifique

Olhando a série e a ACF do dólar:

1. A série é **estacionária** ou **não estacionária**? Justifique com base na ACF.

   _______________________________________________________________

2. Se você quisesse usar um modelo ARIMA, precisaria **transformar** essa série? Como?

   _(dica: diferenciar a série é uma técnica comum)_
   _______________________________________________________________

---

## 🐞 DEBUG CHALLENGE 1

O código abaixo **deveria** decompor a série do IPCA em tendência + sazonalidade + resíduo. Mas **tem um erro**. A saída fica estranha.

**Sua tarefa:**

1. Execute o código como está.
2. Observe o gráfico da sazonalidade — ela parece fazer sentido?
3. **Conserte** o parâmetro errado.
4. Explique o que estava errado.

In [ ]:
# Carregando IPCA
url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2015&dataFinal=31/12/2024"
ipca = pd.read_json(url_ipca)
ipca['data'] = pd.to_datetime(ipca['data'], format='%d/%m/%Y')
ipca = ipca.set_index('data')
ipca.columns = ['IPCA']

print(f"Frequência dos dados: mensal")
print(f"{len(ipca)} observações")

# 🐞 BUG: O período está errado!
decomp = seasonal_decompose(ipca['IPCA'], model='additive', period=4)

fig = decomp.plot()
fig.set_size_inches(10, 8)
plt.tight_layout()
plt.show()

### ✏️ Seu diagnóstico

**Antes de consertar, responda:**

1. Que componente ficou com aparência estranha? ___________________

2. Qual é o período "natural" para uma série **mensal** com sazonalidade anual? ___________

### Agora conserte:

In [ ]:
# SEU CÓDIGO CORRIGIDO:
decomp = seasonal_decompose(ipca['IPCA'], model='additive', period=___)  # preencha!

fig = decomp.plot()
fig.set_size_inches(10, 8)
fig.suptitle('Decomposição corrigida — IPCA', y=1.02)
plt.tight_layout()
plt.show()

### ✏️ Explique o bug

O que significa o parâmetro `period` em `seasonal_decompose`? Por que `4` não funciona para dados mensais?

_Sua explicação (3-4 linhas):_

_______________________________________________________________
_______________________________________________________________
_______________________________________________________________

---

## 2. Linear × Não Linear

- **Série linear:** a tendência é bem representada por uma **reta**.
- **Série não linear:** a tendência é uma **curva** (exponencial, logarítmica, polinomial…).

### 2.1 Simulação

In [ ]:
np.random.seed(123)
t = np.arange(1, 101)

# Série linear: y = 0.5 * t + ruído
serie_lin = 0.5 * t + np.random.normal(0, 2, 100)

# Série não linear: y = 0.05 * t² + ruído
serie_nlin = 0.05 * t**2 + np.random.normal(0, 20, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Linear: ajuste reta
ax1.plot(t, serie_lin, color='steelblue', label='Dados')
slope, intercept, *_ = linregress(t, serie_lin)
ax1.plot(t, slope*t + intercept, 'r--', linewidth=2, label='Ajuste linear')
ax1.set_title('Série Linear'); ax1.legend()

# Não linear: suavização LOWESS
ax2.plot(t, serie_nlin, color='darkgreen', label='Dados')
suav = lowess(serie_nlin, t)
ax2.plot(suav[:,0], suav[:,1], color='orange', linewidth=2, linestyle='--', label='LOWESS')
ax2.set_title('Série Não Linear'); ax2.legend()

plt.tight_layout()
plt.show()

### 🧠 Observe

- Na **linear**, a reta (vermelha) passa bem pela nuvem de pontos.
- Na **não linear**, uma reta seria um **péssimo ajuste** — a curva LOWESS (laranja) captura a forma real.

### 2.2 O truque do logaritmo

Algumas séries com crescimento **exponencial** podem ser "linearizadas" tirando o **log**. Vamos ver com a famosa série AirPassengers.

In [ ]:
url_air = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = pd.read_csv(url_air, parse_dates=['Month'], index_col='Month')

log_air = np.log(air['Passengers'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(air.index, air['Passengers'], color='blue')
ax1.set_title('AirPassengers — original (não linear)')
ax1.set_ylabel('Passageiros')

ax2.plot(log_air.index, log_air, color='darkgreen')
ax2.set_title('log(AirPassengers) — agora linear!')
ax2.set_ylabel('log(Passageiros)')

plt.tight_layout()
plt.show()

### ✏️ Por que o log funciona?

Se `y = a · bᵗ` (crescimento exponencial), então `log(y) = log(a) + t · log(b)` — que é **linear em t**.

**Pergunta:** Qual seria o análogo para uma série que cresce como `y = a · t²`?

_(dica: raiz quadrada)_
_______________________________________________________________

---

## 3. Sazonal × Não Sazonal

Séries **sazonais** têm padrões que **se repetem em intervalos regulares conhecidos** (mensal, semanal, anual…).

Vamos investigar **duas séries brasileiras reais** e classificar cada uma.

In [ ]:
# Já carregamos o IPCA acima. Agora carregamos também a taxa de desemprego PNADC
# Série 24369 = Taxa de desocupação (PNADC trimestral, mas reportada mensalmente)
url_desemp = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.24369/dados?formato=json&dataInicial=01/01/2015&dataFinal=31/12/2024"

desemp = pd.read_json(url_desemp)
desemp['data'] = pd.to_datetime(desemp['data'], format='%d/%m/%Y')
desemp = desemp.set_index('data')
desemp.columns = ['Desemprego (%)']

print(f"Desemprego: {len(desemp)} observações")
desemp.tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(ipca.index, ipca['IPCA'], color='darkblue')
ax1.set_title('IPCA mensal')
ax1.set_ylabel('% ao mês')

ax2.plot(desemp.index, desemp['Desemprego (%)'], color='darkred')
ax2.set_title('Taxa de Desemprego (Brasil)')
ax2.set_ylabel('% da força de trabalho')

plt.tight_layout()
plt.show()

### ✏️ Predição

Olhando só o gráfico (sem fazer decomposição):

- **IPCA:** tem sazonalidade evidente? _____ (S/N)
- **Desemprego:** tem sazonalidade evidente? _____ (S/N)

Agora vamos confirmar com a **ACF** — picos regulares em lags fixos indicam sazonalidade.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(ipca['IPCA'], ax=ax1, lags=36, title='ACF — IPCA')
plot_acf(desemp['Desemprego (%)'], ax=ax2, lags=36, title='ACF — Desemprego')

plt.tight_layout()
plt.show()

### 🔍 Interprete

1. **IPCA:** você vê picos em múltiplos de 12 na ACF? O que isso sugere? _______________________

2. **Desemprego:** o decaimento é rápido ou lento? A série parece estacionária? _______________________

3. A taxa de desemprego tem **ciclo econômico** (recessão × expansão), mas isso é **ciclo**, não **sazonalidade**. Qual a diferença prática?
   _______________________________________________________________

---

## 🐞 DEBUG CHALLENGE 2

O código abaixo tenta plotar a média móvel de 12 meses do IPCA, mas **o resultado fica vazio** (gráfico em branco ou erro).

Rode, identifique o problema e conserte.

In [ ]:
# 🐞 Este código tem um bug
try:
    ipca_valores = ipca['IPCA'].tolist()  # converte para lista Python

    # Tenta calcular média móvel na lista
    mm = ipca_valores.rolling(window=12).mean()

    plt.plot(mm)
    plt.title('Média Móvel 12m do IPCA')
    plt.show()
except Exception as e:
    print(f"❌ Erro: {type(e).__name__}: {e}")

### ✏️ Diagnóstico

1. Qual foi a mensagem de erro? _______________________________________

2. Por que o erro aconteceu? (dica: listas Python têm método `.rolling()`?)
   _______________________________________________________________

### Conserte:

In [ ]:
# SEU CÓDIGO CORRIGIDO
# Dica: mantenha os dados como pd.Series, não converta para lista

mm = ____________________________________________  # preencha

plt.plot(mm)
plt.title('Média Móvel 12m do IPCA — corrigido')
plt.xlabel('Ano'); plt.ylabel('% ao mês')
plt.show()

---

## 4. Univariada × Multivariada

- **Univariada:** você acompanha **uma única variável** ao longo do tempo.
- **Multivariada:** você acompanha **duas ou mais variáveis relacionadas**.

Em multivariada, é possível investigar **correlações temporais** — por exemplo: quando o dólar sobe, a inflação também sobe?

### 4.1 Análise multivariada: IPCA × Dólar × Desemprego

In [ ]:
# Vamos juntar as 3 séries em um único DataFrame
# Primeiro alinhamos pela data (mensal)

df_multi = pd.concat([
    ipca['IPCA'],
    dolar['USD/BRL'],
    desemp['Desemprego (%)']
], axis=1, join='inner')

df_multi.columns = ['IPCA', 'USD/BRL', 'Desemprego']
df_multi = df_multi.dropna()

print(f"Período alinhado: {df_multi.index.min().date()} a {df_multi.index.max().date()}")
df_multi.tail()

In [ ]:
# Plot múltiplo
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df_multi.index, df_multi['IPCA'], color='darkblue')
axes[0].set_ylabel('IPCA (%)')
axes[0].set_title('Três séries macroeconômicas brasileiras')

axes[1].plot(df_multi.index, df_multi['USD/BRL'], color='darkgreen')
axes[1].set_ylabel('USD/BRL (R$)')

axes[2].plot(df_multi.index, df_multi['Desemprego'], color='darkred')
axes[2].set_ylabel('Desemprego (%)')
axes[2].set_xlabel('Ano')

plt.tight_layout()
plt.show()

### 4.2 Correlação entre séries

In [ ]:
corr = df_multi.corr()
print("Matriz de correlação:")
print(corr.round(3))

In [ ]:
# Visualização da matriz de correlação
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black')

plt.title('Correlação entre IPCA, Dólar e Desemprego')
plt.colorbar(im)
plt.tight_layout()
plt.show()

### 🧠 Interprete os resultados

1. **Dólar × IPCA:** a correlação é positiva ou negativa? Isso faz sentido economicamente? (quando o real desvaloriza, importações ficam mais caras)

   _______________________________________________________________

2. **Desemprego × IPCA:** qual o sinal da correlação? Isso contradiz ou confirma a teoria econômica (Curva de Phillips)?

   _______________________________________________________________

3. **Cuidado:** correlação em séries temporais pode ser **espúria** se ambas tiverem tendência. Para análise rigorosa, o que deveríamos fazer antes de calcular correlação?

   _(dica: tornar as séries estacionárias primeiro)_
   _______________________________________________________________

---

## 5. Síntese — diagnóstico completo

Preencha a tabela abaixo **com base em tudo que vimos**:

| Série | Estacionária? | Linear? | Sazonal? | Tipo |
|-------|---------------|---------|----------|------|
| IPCA mensal | ___ | ___ | ___ | univariada |
| USD/BRL | ___ | ___ | ___ | univariada |
| Desemprego | ___ | ___ | ___ | univariada |
| IPCA + USD/BRL + Desemprego | — | — | — | multivariada |

---

## 6. Exercício final — sua escolha de série

A API do Banco Central tem **milhares de séries econômicas brasileiras**. Algumas sugestões:

| Código | Série |
|---|---|
| 1207 | Vendas reais no comércio varejista |
| 21619 | Produção industrial mensal |
| 4380 | PIB mensal (índice) |
| 433  | IPCA |
| 3698 | Dólar mensal |

**Escolha UMA série** (qualquer código do BCB) e faça o diagnóstico completo no código abaixo.

In [ ]:
# ESCOLHA SUA SÉRIE
codigo = ___     # preencha com o código da série
nome = "___"    # nome descritivo

url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial=01/01/2015&dataFinal=31/12/2024"
df = pd.read_json(url)
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df = df.set_index('data')
df.columns = [nome]

# 1. Plot da série

# 2. ACF

# 3. Decomposição

# 4. Classificação (escreva em markdown abaixo)


### 📋 Seu relatório

**Série escolhida:** _______________

**Observações visuais:**
_______________________________________________________________

**Estacionária?** _______ Por quê? _______________

**Linear?** _______ Por quê? _______________

**Sazonal?** _______ Qual período? _______

**Modelo aditivo ou multiplicativo?** _______ Por quê? _______________